In [ ]:
#crow2/phangs/catelogs
#crow2/phangs/hst
import os
from Functions import *
os.chdir('/project/galaxies') #TJ change working directory to be the parent directory
_, filter_files = collect_M51_image_and_filter_files(filter_directory, image_directory)
from astropy.table import Table
import glob
from matplotlib.path import Path
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np
#TJ To create table of galaxies, run tjuchau/projects/EW_vs_Age/Create_cluster_table.py
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')

#table = table[np.isfinite(table['EW_187'])]
m51_clusters = Table.read('/project/galaxies/tjuchau/data_files/misc_data/clusters.csv')
#m51_clusters1 = Table.read('/project/galaxies/tjuchau/data_files/misc_data/clusters_test.csv')
table.sort('best.sfh.age')
#table = table[table['best.sfh.age'] > 0]
table

In [ ]:
show_images?

In [ ]:
for row in table[~np.isfinite(table['EW_187'])]:
    try:
        show_images(['/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc.fits',
    '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits',
        '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits',
        '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f300m_i2d_anchor.fits'], [row['ra'], row['dec']], 0.5*u.arcsec, ncols=4)
    except:
        print('')

In [ ]:
def plot_EWvAge(table, mass_bins = 8):
    fontsize_sm = 25
    fontsize_lg = 50
    cols = 2
    rows = int(np.ceil(mass_bins/cols))

    masses = np.logspace(np.log10(min(table['best.stellar.m_gas'] + table['best.stellar.m_star'])),
    np.log10(max(table['best.stellar.m_gas'] + table['best.stellar.m_star'])),
    mass_bins+1)
    fig, axes = plt.subplots(rows, cols, figsize=(40, 30))
    axes = axes.ravel()
    for i in range(mass_bins+1):
        if i == mass_bins:
            continue
        print(f'mass range from {masses[i]} to {masses[i+1]}')
        chunk = table[((table['best.stellar.m_gas'] + table['best.stellar.m_star']) > masses[i]) & ((table['best.stellar.m_gas'] + table['best.stellar.m_star']) < masses[i+1])]
        axes[i].set_title(f'{masses[i]:.0f} to {masses[i+1]:.0f} solar masses', fontsize = fontsize_sm)
        axes[i].set_yscale('log')
        axes[i].set_xscale('log')
        scatter = axes[i].scatter(chunk['best.sfh.age'], chunk['EW_187'])
        axes[i].tick_params(axis='x', which='minor', width=2, length=10, right=True, top=True, direction='in',
                        labelsize=fontsize_sm)
        axes[i].tick_params(axis='x', which='major', width=3, length=15, right=True, top=True, direction='in',
                            labelsize=fontsize_sm)
        axes[i].tick_params(axis='y', which='both', width=3, length=15, right=True, top=True, direction='in',
                            labelsize=fontsize_sm)
        if i < mass_bins -2:
            axes[i].set_xlabel('')
        else:
            axes[i].set_xlabel('Age (Myr)', fontsize=40)
        if np.mod(i,2) == 0:
            axes[i].set_ylabel(f'Pa-alpha EW (A)', fontsize=35)
        else:
            axes[i].set_ylabel('', fontsize=35)


    return masses
plot_EWvAge(table)

In [ ]:
test = glob.glob('/project/galaxies/tjuchau/data_files/misc_data/*')
temp = []
for row in my_table[:10]:
    temp.append(test)
my_table.add_column(temp, name = 'testing')
my_table.write('/project/galaxies/tjuchau/data_files/misc_data/temp_data/test_save_table.csv', format='csv', overwrite=True)

my_table

In [ ]:
for i, row in enumerate(table[:20]):
    gal_name = row['galaxy']
    loc = [row['ra'], row['dec']]
    radius = row['radius']*u.arcsec
    if gal_name == "M51":
        image_files = ['/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits',
        '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits',
        '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f300m_i2d_anchor.fits',
        '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits']
    else:
        image_files = glob.glob(f'tjuchau/data_files/JWST/images/{gal_name.upper()}/*')
    print(i)
    try:
        show_images(image_files, loc, radius, cmap = 'rainbow', ncols=4)

    except:
        print('Image does not cover this location')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, unique

# Columns
gal = table['galaxy']
age = table['best.sfh.age']
ew  = table['EW_187']/(table['best.stellar.m_gas']+table['best.stellar.m_star'])

# Container lists
gal_list = []
age_list = []
ew_mean = []
ew_std = []

# Loop through galaxies
for g in np.unique(gal):

    gal_mask = gal == g
    ages_in_gal = np.unique(age[gal_mask])

    for a in ages_in_gal:

        mask = (gal == g) & (age == a)

        ew_vals = ew[mask]

        if len(ew_vals) > 0:
            gal_list.append(g)
            age_list.append(a)
            ew_mean.append(np.mean(ew_vals))
            ew_std.append(np.std(ew_vals))

gal_list = np.array(gal_list)
age_list = np.array(age_list)
ew_mean = np.array(ew_mean)
ew_std = np.array(ew_std)

# Plot
plt.figure()

for g in np.unique(gal_list):

    mask = gal_list == g

    plt.errorbar(
        age_list[mask],
        ew_mean[mask],
        yerr=ew_std[mask],
        fmt='o',
        label=g,
        capsize=3
    )

plt.xlabel("Cluster Age")
plt.ylabel("Paα Equivalent Width")

plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

x = table['best.sfh.age']
y = table['EW_658']/table['EW_187']
plt.scatter(x, y, s = 5, color = 'blue')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(x.name)
plt.ylabel('NII/Pa-a')
plt.show()

x = table['best.sfh.age']
y = table['EW_658']
plt.scatter(x, y, s = 5, color = 'blue')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(x.name)
plt.ylabel('NII')
plt.show()

x = table['best.sfh.age']
y = (table['EW_187']/table['EW_658'])/(table['best.stellar.m_gas'])
plt.scatter(x, y, s = 5, color = 'blue')
plt.xscale('log')
plt.xlabel(x.name)
plt.ylabel('Pa-a/mass')
plt.show()

x = table['best.sfh.age']
y = (table['EW_187'])
plt.scatter(x, y, s = 5, color = 'blue')
plt.xscale('log')
plt.xlabel(x.name)
plt.ylabel('Pa-a')
plt.show()

In [ ]:
import pickle
galaxy_tables['ngc1433'][7]['radius'] = 0.4
with open("/project/galaxies/tjuchau/data_files/misc_data/saved_data/Kiana_galaxies.pkl", "wb") as file:
    pickle.dump(galaxy_tables, file)

In [ ]:
young_clusters = kiana_table[kiana_table['EW_658'] > 8e-9]
image_files = glob.glob(f'tjuchau/data_files/JWST/images/NGC1433/*')
ages = []
ews = []
for row in young_clusters:
    ra = row['ra']
    dec = row['dec']
    age = row['best.sfh.age']
    ages.append(age)
    pa_EW = get_EW_using_filters(image_files[1], [image_files[0], image_files[2]], [ra,dec], 0.5*u.arcsec).value
    ews.append(pa_EW)
    print(f'Age : {age}, Pa-EW : {pa_EW}')
    try:
        show_images(image_files, [ra,dec], 0.5*u.arcsec, cmap = 'rainbow')
    except:
        print('Image does not cover this location')
plt.scatter(ages, ews)
plt.show()

In [ ]:
name = 'ngc1433'
ngc1433_bad = [0,4,6,9,10]
ngc1433_maybe = [7]
ngc1433_good = np.setdiff1d(np.arange(len(galaxy_tables[name])), np.concatenate([ngc1433_bad, ngc1433_maybe]))
image_files = glob.glob(f'tjuchau/data_files/JWST/images/{name.upper()}/*')
for i, row in enumerate(galaxy_tables[name][:100]):
    if i in ngc1433_good:
        location = row['location']
        print(i)
        show_images(image_files, location, row['radius']*u.arcsec, cmap = 'rainbow')

In [ ]:
for i, row in enumerate(galaxy_tables[name][ngc1433_bad]):
    location = row['location']
    print(ngc1433_bad[i])
    show_images(image_files, location, 0.3*u.arcsec, cmap = 'rainbow')

In [ ]:
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')
table